# Lab 10: Data File Analyzer (DS-STAR Component)

**Navigation** : [Lab 9 <<](../Day4-Foundations/Lab9-First-ADK-Agent.ipynb) | [Index](../../README.md) | [>> Lab 11](Lab11-Planner-Coder-Loop.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Implémenter le module File Analyzer de DS-STAR
2. Analyser des fichiers hétérogènes (CSV, JSON, Markdown, texte)
3. Extraire des métadonnées structurées automatiquement
4. Générer des résumés intelligents avec le LLM

### Prérequis
- Python 3.10+
- Configuration multi-provider active
- Connaissance de Pandas (Lab 4)

### Durée estimée : 30-40 minutes

## 1. Configuration

In [1]:
import sys
from pathlib import Path

# Add parent directory (AgenticDataScience) to path for config/utils imports
sys.path.insert(0, str(Path().resolve().parent))

import os
import json
import pandas as pd
import numpy as np
from dataclasses import dataclass, asdict

from config import get_settings
from utils import LLMClient

Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openai


## 2. FileMetadata DataClass

In [3]:
@dataclass
class FileMetadata:
    filename: str
    format: str
    size_bytes: int
    num_rows: int = None
    num_columns: int = None
    columns: list = None
    statistics: dict = None
    sample_data: list = None

## 3. FileAnalyzer Class

In [4]:
class FileAnalyzer:
    SUPPORTED = {'.csv': 'csv', '.json': 'json', '.md': 'markdown', '.txt': 'text'}

    def __init__(self, llm_client=None):
        self.llm = llm_client or LLMClient()

    def detect_format(self, path):
        ext = Path(path).suffix.lower()
        return self.SUPPORTED.get(ext, 'unknown')

    def analyze_csv(self, path):
        df = pd.read_csv(path)
        cols = []
        for c in df.columns:
            info = {'name': c, 'dtype': str(df[c].dtype), 'missing_pct': round(df[c].isna().mean()*100, 2)}
            if df[c].dtype in ['int64', 'float64']:
                info['stats'] = {'min': float(df[c].min()), 'max': float(df[c].max()), 'mean': float(df[c].mean())}
            cols.append(info)
        return FileMetadata(
            filename=Path(path).name, format='csv', size_bytes=os.path.getsize(path),
            num_rows=len(df), num_columns=len(df.columns), columns=cols,
            sample_data=df.head(5).to_dict('records')
        )

    def analyze(self, path):
        fmt = self.detect_format(path)
        if fmt == 'csv':
            return self.analyze_csv(path)
        raise ValueError(f"Format non supporte: {fmt}")

    def generate_summary(self, meta):
        lines = [f"Fichier: {meta.filename}", f"Format: {meta.format}", f"Taille: {meta.size_bytes/1024:.1f} KB"]
        if meta.num_rows:
            lines.append(f"Lignes: {meta.num_rows}, Colonnes: {meta.num_columns}")
        if meta.columns:
            for c in meta.columns[:5]:
                lines.append(f"  - {c['name']} ({c['dtype']}): {c.get('missing_pct', 0)}% manquant")
        return "\n".join(lines)

    def generate_llm_summary(self, meta, question=None):
        ctx = self.generate_summary(meta)
        if meta.sample_data:
            ctx += f"\n\nEchantillon:\n{json.dumps(meta.sample_data[:2], indent=2, default=str)[:400]}"
        prompt = f"Analyse ce fichier et resume-le:\n{ctx}\n{f'Question: {question}' if question else ''}"
        return self.llm.generate(prompt, temperature=0.3)

## 4. Tests

In [5]:
# Creation fichier test
import tempfile
test_dir = tempfile.mkdtemp()

df = pd.DataFrame({
    'id': range(1, 101),
    'product': [f'P{i}' for i in range(1, 101)],
    'category': np.random.choice(['A', 'B', 'C'], 100),
    'price': np.random.uniform(10, 500, 100).round(2),
    'qty': np.random.randint(1, 50, 100)
})
df.loc[10:15, 'price'] = np.nan

csv_path = os.path.join(test_dir, 'products.csv')
df.to_csv(csv_path, index=False)
print(f'CSV cree: {csv_path}')

CSV cree: C:\Users\jsboi\AppData\Local\Temp\tmpbtwg0yvk\products.csv


Test de l'analyseur de fichiers.

In [6]:
# Test analyseur
analyzer = FileAnalyzer()
meta = analyzer.analyze(csv_path)
print(analyzer.generate_summary(meta))

Fichier: products.csv
Format: csv
Taille: 1.9 KB
Lignes: 100, Colonnes: 5
  - id (int64): 0.0% manquant
  - product (str): 0.0% manquant
  - category (str): 0.0% manquant
  - price (float64): 6.0% manquant
  - qty (int64): 0.0% manquant


Detail des colonnes detectees.

In [7]:
# Colonnes detail
for c in meta.columns:
    print(f"{c['name']}: {c['dtype']}, {c.get('missing_pct', 0)}% manquant")
    if 'stats' in c:
        print(f"  -> min={c['stats']['min']:.1f}, max={c['stats']['max']:.1f}, mean={c['stats']['mean']:.1f}")

id: int64, 0.0% manquant
  -> min=1.0, max=100.0, mean=50.5
product: str, 0.0% manquant
category: str, 0.0% manquant
price: float64, 6.0% manquant
  -> min=13.5, max=489.3, mean=261.4
qty: int64, 0.0% manquant
  -> min=2.0, max=49.0, mean=26.5


## 5. Resume LLM

In [8]:
# Resume intelligent
summary = analyzer.generate_llm_summary(meta)
print(summary)

Le fichier `products.csv` est un fichier CSV de taille 1.9 KB contenant 100 lignes et 5 colonnes. Voici un résumé des colonnes :

1. **id (int64)** : Cette colonne contient des identifiants uniques pour chaque produit. Il n'y a pas de valeurs manquantes dans cette colonne.

2. **product (str)** : Cette colonne contient les noms des produits. Il n'y a pas de valeurs manquantes dans cette colonne.

3. **category (str)** : Cette colonne indique la catégorie à laquelle chaque produit appartient. Il n'y a pas de valeurs manquantes dans cette colonne.

4. **price (float64)** : Cette colonne contient le prix de chaque produit. Il y a 6% de valeurs manquantes dans cette colonne, ce qui signifie que certains produits n'ont pas de prix renseigné.

5. **qty (int64)** : Cette colonne indique la quantité disponible pour chaque produit. Il n'y a pas de valeurs manquantes dans cette colonne.

L'échantillon fourni montre deux produits appartenant à la même catégorie "C", avec des prix et des quantités

Generation d'un resume intelligent guide par le LLM.

In [9]:
# Resume guide
q = "Quelles analyses puis-je faire sur ces donnees?"
print(f"Question: {q}\n")
result = analyzer.generate_llm_summary(meta, q)
print(result)

Question: Quelles analyses puis-je faire sur ces donnees?



Avec le fichier `products.csv`, vous pouvez réaliser plusieurs types d'analyses pour obtenir des informations utiles sur vos produits. Voici quelques suggestions :

1. **Analyse des prix manquants**:
   - Évaluer l'impact des 6% de valeurs manquantes dans la colonne `price` sur vos analyses.
   - Explorer des méthodes pour imputer ces valeurs manquantes, par exemple en utilisant la moyenne ou la médiane des prix par catégorie.

2. **Statistiques descriptives**:
   - Calculer des statistiques de base comme la moyenne, la médiane, le minimum et le maximum des prix et des quantités (`qty`).
   - Identifier les catégories de produits les plus et les moins chères.

3. **Distribution des produits par catégorie**:
   - Compter le nombre de produits dans chaque catégorie pour identifier les catégories les plus populaires.
   - Analyser la répartition des prix et des quantités au sein de chaque catégorie.

4. **Analyse des ventes potentielles**:
   - Calculer la valeur totale potentielle des ve

## 6. Resume du Lab### Points cles1. FileAnalyzer: Analyse automatique de fichiers2. Metadonnees: Extraction de schemas et stats3. Resume LLM: Contextualisation pour le Planner### Prochaine etape- Lab 11: Boucle Planner-Coder-Verifier

In [10]:
# Cleanup
import shutil
shutil.rmtree(test_dir)
print('Done')

Done


## Exercice

À vous d'étendre le FileAnalyzer pour gérer un nouveau format de fichier !


In [11]:
# Exercice : Etendez le FileAnalyzer pour gerer les fichiers JSON
# 1. Ajoutez une methode analyze_json
# 2. Testez-la sur un fichier JSON de test

import json
import tempfile

# Creation d'un fichier JSON de test
exercise_dir = tempfile.mkdtemp()
test_json_path = os.path.join(exercise_dir, 'test_data.json')
test_data = {
    'products': [
        {'id': 1, 'name': 'Laptop', 'price': 999.99, 'stock': 15},
        {'id': 2, 'name': 'Mouse', 'price': 19.99, 'stock': 50},
        {'id': 3, 'name': 'Keyboard', 'price': 49.99, 'stock': 30}
    ],
    'metadata': {
        'source': 'inventory_system',
        'last_updated': '2026-03-13'
    }
}

with open(test_json_path, 'w') as f:
    json.dump(test_data, f)

# TODO: Implementez la classe ExtendedFileAnalyzer
# Elle doit heriter de FileAnalyzer et ajouter une methode analyze_json
class ExtendedFileAnalyzer(FileAnalyzer):
    def analyze_json(self, path):
        """Analyse un fichier JSON et extrait les metadonnees."""
        # TODO: Ouvrez et chargez le fichier JSON
        # Indice: with open(path, 'r') as f: data = json.load(f)
        data = None  # Remplacez None

        # TODO: Comptez les cles au niveau racine
        root_keys = None  # Indice: list(data.keys())

        # TODO: Trouvez la plus grande liste dans les valeurs
        # Indice: parcourez data.items(), testez isinstance(value, list)
        max_list = 0
        list_name = None
        # ... votre code ici

        # TODO: Retournez un FileMetadata avec les informations extraites
        return None  # Remplacez par FileMetadata(...)

# TODO: Testez l'analyseur etendu
# extended_analyzer = ExtendedFileAnalyzer()
# json_meta = extended_analyzer.analyze_json(test_json_path)
# print(f"Fichier: {json_meta.filename}")
# print(f"Cles racine: {json_meta.columns}")

# Nettoyage
# os.remove(test_json_path)

print("Exercice a completer")

Exercice a completer
